In [1]:
!pip install -q kaggle

In [2]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"mehedihasan1613","key":"7b4a64536351229c2cb784ba22425547"}'}

In [3]:
! mkdir ~/.kaggle
! cp kaggle.json ~/.kaggle/
! chmod 600 ~/.kaggle/kaggle.json

In [4]:
! kaggle datasets download dadalyndell/elon-musk-tweets-2010-to-2025-march

Dataset URL: https://www.kaggle.com/datasets/dadalyndell/elon-musk-tweets-2010-to-2025-march
License(s): other


In [5]:
!unzip elon-musk-tweets-2010-to-2025-march.zip

Archive:  elon-musk-tweets-2010-to-2025-march.zip
  inflating: all_musk_posts.csv      
  inflating: musk_quote_tweets.csv   


In [6]:
!pip install emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 590.6/590.6 kB 11.2 MB/s eta 0:00:00


**Importing packages**

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
from nltk.corpus import stopwords
import emoji
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')
nltk.download('stopwords')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [8]:

df_posts = pd.read_csv('all_musk_posts.csv')

print(df_posts.info())

print("Date range:", df_posts['createdAt'].min(), "to", df_posts['createdAt'].max())

print(df_posts['fullText'].sample(5))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55099 entries, 0 to 55098
Data columns (total 24 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   id                        55099 non-null  int64  
 1   url                       55099 non-null  object 
 2   twitterUrl                55099 non-null  object 
 3   fullText                  55099 non-null  object 
 4   retweetCount              55009 non-null  float64
 5   replyCount                54297 non-null  float64
 6   likeCount                 55009 non-null  float64
 7   quoteCount                54271 non-null  float64
 8   viewCount                 34455 non-null  float64
 9   createdAt                 55099 non-null  object 
 10  bookmarkCount             54271 non-null  float64
 11  isReply                   54271 non-null  object 
 12  inReplyToId               39497 non-null  float64
 13  conversationId            54297 non-null  float64
 14  inRepl

<ipython-input-8-0cb41105a1dd>:1: DtypeWarning: Columns (11,16,17,18,19) have mixed types. Specify dtype option on import or set low_memory=False.
  df_posts = pd.read_csv('all_musk_posts.csv')


In [9]:
df_quote = pd.read_csv('musk_quote_tweets.csv')
print(df_quote.info())

print("Date range:", df_quote['orig_tweet_created_at'].min(), "to", df_quote['orig_tweet_created_at'].max())

print(df_quote['orig_tweet_text'].sample(5))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7273 entries, 0 to 7272
Data columns (total 21 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   orig_tweet_id              7273 non-null   int64  
 1   orig_tweet_created_at      7273 non-null   object 
 2   orig_tweet_text            7273 non-null   object 
 3   orig_tweet_url             7273 non-null   object 
 4   orig_tweet_twitter_url     7273 non-null   object 
 5   orig_tweet_username        7273 non-null   object 
 6   orig_tweet_retweet_count   7273 non-null   int64  
 7   orig_tweet_reply_count     7273 non-null   int64  
 8   orig_tweet_like_count      7273 non-null   int64  
 9   orig_tweet_quote_count     7273 non-null   int64  
 10  orig_tweet_view_count      6760 non-null   float64
 11  orig_tweet_bookmark_count  7273 non-null   int64  
 12  musk_tweet_id              7273 non-null   int64  
 13  musk_quote_tweet           7273 non-null   objec

In [13]:
def clean_tweet(tweet):
    tweet = re.sub(r'http\S+|www\S+|https\S+', '', tweet, flags=re.MULTILINE)  #Removing Urls
    tweet = re.sub(r'@\w+|#\w+', '', tweet) #Removing mentions
    tweet = re.sub(r'[^a-zA-Z\s]', '', tweet) #Removing special characters and number
    tweet = tweet.lower() #converting everyting into lowercase letters
    tweet = re.sub(r'\s+', ' ', tweet).strip() #remove whitespaces
    stop_words = set(stopwords.words('english')) #common english words with no significant meaning
    tweet = ' '.join([word for word in tweet.split() if word not in stop_words])
    tweet = emoji.demojize(tweet, delimiters=(' ', ' ')) #Replace emojis with words
    return tweet

In [14]:
df_posts['cleaned_text'] = df_posts['fullText'].apply(clean_tweet)
df_posts = df_posts[df_posts['cleaned_text'].str.len() > 0]

lemmatizer = WordNetLemmatizer()
df_posts['cleaned_text'] = df_posts['cleaned_text'].apply(
    lambda x: ' '.join([lemmatizer.lemmatize(word) for word in x.split()])
)

In [15]:
print("Original Tweet:", df_posts['fullText'].iloc[0])
print("Cleaned Tweet:", df_posts['cleaned_text'].iloc[0])

Original Tweet: RT @einarvollset: I read @paulg’s  “How to Make Wealth” in Hackers &amp; Painters when I was in my mid twenties. It’s one of those pieces that…
Cleaned Tweet: rt read make wealth hacker amp painter mid twenty one piece
